# This is an adapted notebook so Valerie can create baseline spot detections to use for the spotiflow refinement project.

### Use within the LSM-CME-ANALYSIS workspace & cme_pipeline environment
### Spots are detected in this notebook. The input file is expected to be in the zarr format 

In [1]:
import pandas as pd
import time
import os
import sys
import zarr
import napari 
import dask.array as da 

pythonPackagePath = os.path.abspath('../src/')
sys.path.append(pythonPackagePath)
from parallel import Detector
from gaussian_visualization import visualize_3D_gaussians

### Do not change the code in cell below 

In [2]:
# This assumes that your notebook is inside 'Jupyter Notebooks', which is at the same level as 'test_data'
base_dir = os.path.join(os.path.dirname(os.path.abspath("__file__")), '..', 'movie_data')
# base_dir = os.path.join(os.path.dirname(os.path.abspath("__file__")), '..', 'test_movie_1')

zarr_directory = 'zarr_file/all_channels_data'
zarr_full_path = os.path.join(base_dir, zarr_directory)

save_directory = 'datasets'
save_directory_full = os.path.join(base_dir, save_directory)

## Follow the Instructions below to run through the notebook properly 

This notebook detects spots on your movie. The movie should be a zarr object; if it's not, run Final/Data Preparation/full_movie_to_zarr.ipynb

**Parameters to adjust below** 

* **channel_to_detect**: Which channel will be tracked? This should be the channel with the longest tracks (i.e. AP2). Options are channel 1, 2, or 3.

* **threshold_intensity**: What intensity value distinguishes background from signal? Open up a frame of the movie in Fiji or napari (at the end of this notebook) and mouse over different pixels to figure out this threshold value.

* **all_frames**: When initially optimizing, set this to false and set number_frames_to_detect to 2, in order to run detection on only two time points. This will speed up diagnosing detection quality at the end of the notebook.

Additional parameters for optimization:

* **dist_between_spots**: this distance divided by 2 is the minimum distance that should exist between spots in pixels. For example if you set this to 10 then all spots within 5 pixels of the center of your spot will not be detected. 
* **sigma_estimations**: The expected radius of our spots, in pixels, as [spread_in_z, spread_in_y, spread_in_x]. You can measure the width of a spot in Fiji and divide by two.
* **n_jobs**: The number of CPUs to use for detections. You can set it to -1 and it will use all of your machine's CPUs but one for processing. 

* **number_frames_to_detect**: the number of frames to process. This can be useful when you just want to test your parameters selected for the Detector object like spot_intensity, dist_between_spots and sigma_estimates. 



## Set all parameters in the below cell 

In [3]:
#refer to the above cell for explanation of each parameter 
channel_to_detect = 1 #fixed because I only have one channel
threshold_intensity = 150
all_frames = True

dist_between_spots = 6
sigma_estimations = [2,2,2]
n_jobs = -1
number_frames_to_detect = 1 #fixed because I'm only looking at one frame

In [4]:
#Import the zarr file by adding file path in read mode
z2 = zarr.open(zarr_full_path, mode='r')
frames = z2.shape[0]
print(f'the number of frames are {frames}')
z2.info

the number of frames are 1


Type,zarr.core.Array
Data type,uint16
Shape,"(1, 1, 31, 175, 169)"
Chunk shape,"(1, 1, 31, 175, 169)"
Order,C
Read-only,True
Compressor,"Blosc(cname='lz4', clevel=5, shuffle=SHUFFLE, blocksize=0)"
Store type,zarr.storage.DirectoryStore
No. bytes,1833650 (1.7M)
No. bytes stored,921774 (900.2K)
Storage ratio,2.0


## In the below cell Detector object is initilized to perform detection. More details on the Detector object can be attained by the following line of code: 
**copy and paste in a new cell**

?Detector

In [5]:
detector = Detector(zarr_obj = z2, 
                    save_directory = save_directory_full, 
                    spot_intensity = threshold_intensity, 
                    dist_between_spots = dist_between_spots, 
                    sigma_estimations = sigma_estimations, n_jobs = n_jobs, channel_to_detect = channel_to_detect)

In [6]:
#the following function returns the dataframe and also saves it to the provided path in pkl format
#set all_frames = True, to process all the time frames 
#max_frames is useful when you just want to perform detection on a subset of frames. 
#Note: when all_frames= True then max_frames is ignored 
df = detector.run_parallel_frame_processing(max_frames = number_frames_to_detect, all_frames = all_frames)

Processing frames:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/valerie/LLSM-CME-ANALYSIS/Final/src/gaussian_fitting.py:82: OptimizeWarning: Covariance of the parameters could not be estimated
  z_popt, z_pcov = curve_fit(gaussian_1d_output, z_range, z_data, bounds = ([peak-width_parameters,z_center-width_parameters,-np.inf],[peak+width_parameters,z_center+width_parameters,np.inf]))
Processing frames: 100%|██████████| 1/1 [00:02<00:00,  2.27s/it]

the number of times the gaussian fitting worked was 78 and the number of times the gaussian did not fit was 0


# Visualising the Output
## Labels are only for time frame 0, for all z slices 

## Below you can see detected spots as masks on the original image and can adjust detection parameters if you think spots are not detected correctly 

### Once you are in the napari viewer you should adjust the contrast and the opacity to make sure both the masks and the raw movie is visible properly.  

In [7]:
# z2.info

In [8]:
# Make a mask of the first time point of the detections
# masks = visualize_3D_gaussians(zarr_obj = z2, gaussians_df = df[df['frame'] == 0])
# masks = visualize_3D_gaussians(zarr_obj = z2, gaussians_df = df)

#Drop the points that are clearly outside of bounds in z
#dropping spots out of bounds of z axis 
condition_8 = df['mu_z'] >= 0
condition_9 = df['mu_z'] <= z2.shape[2]

# Combine the conditions using logical AND (&)
cleaned_spots_df = df[condition_8 & condition_9].reset_index(drop = True)

# Create an array of the points
points = cleaned_spots_df[cleaned_spots_df['frame'] == 0][['mu_z', 'mu_y', 'mu_x']].values
spots = df[df['frame'] == 0][['mu_z', 'mu_y', 'mu_x']].values   

# Create a napari viewer
viewer = napari.Viewer()

#open the zarr file in read mode
dask_array = da.from_zarr(z2)

# first time point of the zarr file and the channel to detect
#the axis arrangement is (t,c,z,y,x)

dask_array_slice = dask_array[0,channel_to_detect-1,:,:,:]

# Add the 3D stack to the viewer
layer_raw = viewer.add_image(dask_array_slice, name='fluorescence', interpolation3d = 'nearest', blending = 'additive', colormap = 'magenta')

# layer_mask = viewer.add_image(masks, name = 'detections mask')
# layer_mask = viewer.add_image(masks, name = 'detections', interpolation3d = 'nearest', blending = 'additive', colormap = 'green')
points_layer = viewer.add_points(points, size=3, face_color='green', name='points', symbol='ring')
spots_layer = viewer.add_points(spots, size=3, face_color='blue', name='spots', symbol='ring')


#other useful parameters 
#color_map = list
#contrast_limits = list of list 

# Add Bounding Box
layer_raw.bounding_box.visible = True


---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File ~/miniconda3/envs/cme_pipeline/lib/python3.10/site-packages/napari/_qt/threads/status_checker.py:114, in StatusChecker.calculate_status(self=<napari._qt.threads.status_checker.StatusChecker object>)
    110     return
    112 try:
    113     # Calculate the status change from cursor's movement
--> 114     res = viewer._calc_status_from_cursor()
        viewer = Viewer(camera=Camera(center=(16.0, 87.0, 84.0), zoom=5.873714285714285, angles=(-9.40407937209373, 7.951386703658792e-16, 90.00000000000017), perspective=0.0, mouse_pan=False, mouse_zoom=True), cursor=Cursor(position=(16.000000000000245, 25.450444342689217, 109.70044068602655), scaled=True, style=<CursorStyle.STANDARD: 'standard'>, size=1.0), dims=Dims(ndim=3, ndisplay=3, order=(0, 1, 2), axis_labels=('0', '1', '2'), rollable=(True, True, True), range=(RangeTuple(start=0.0,

If the detections don't line up well with the spots in the image:
* mouse over the spots in napari to get a sense for the intensity of the spots vs background - use the threshold distinguishing spots from background as threshold_intensity 
* vary the dist_between_spots: if the detections are at a higher density than the visible spots, increase the dist_between_spots. And vice versa, if you see spots at a higher density than detections, lower the dist_between_spots.
* If the detections are missing larger or smaller spots you can try increasing or decreasing the sigma_estimations. 
If you see elongated detections, these will be filtered out in the next notebook.

# move to 02.filtering_spots for next steps 